# 03 · Features — encoding, scaling, engineering

**Dataset:** `data/titanic_clean.csv`.
**Covers:** feature engineering · encoding · scaling · multicollinearity · leakage.
**Time yourself:** ~35 minutes.

The theme of this notebook is **leakage**. Almost every question has a version that
looks right and leaks. If you take one thing away: *split first, fit on train only.*

In [9]:
import os
if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

df = pd.read_csv('data/titanic_clean.csv')
df.columns = df.columns.str.lower()
df.head()

,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


---

## Part A — Feature engineering

### Q1. Create `family_size` (siblings + parents/children + the passenger themself) and a
binary `is_alone`. Then check the survival rate by `family_size` — is the
relationship monotonic?

<details><summary>hint</summary>

`sibsp + parch + 1`. The `+1` is the passenger. Then group by it and look at the shape of the rates, not just whether they go up.

</details>

In [11]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)

print(df.groupby('family_size')['survived'].agg(n='count', rate='mean').round(3))

# Not monotonic -- it's an inverted U. Alone: ~30%. Size 2-4: rises 55% -> 72%. Size 5+:
# collapses to ~0-33% (and the counts there are tiny, so treat those rates with caution).
# Small families helped each other; large families couldn't move together.
# A linear model cannot express this from family_size alone -- it needs the bins or a tree.

               n   rate
family_size            
1            537  0.304
2            161  0.553
3            102  0.578
4             29  0.724
5             15  0.200
6             22  0.136
7             12  0.333
8              6  0.000
11             7  0.000


### Q2. Bin `family_size` into `alone / small / large` — you just saw why. Then compare the
survival rate across the three bins.

<details><summary>hint</summary>

`pd.cut` with bins `[0, 1, 4, 20]` — remember cut is right-inclusive by default, so `(0, 1]` captures exactly family_size == 1.

</details>

In [12]:
df['family_bin'] = pd.cut(df['family_size'], bins=[0, 1, 4, 20],
                          labels=['alone', 'small', 'large'])
print(df.groupby('family_bin', observed=True)['survived'].agg(n='count', rate='mean').round(3))

# alone ~30%, small ~58%, large ~16%. Now a linear model can use it too.

              n   rate
family_bin            
alone       537  0.304
small       292  0.579
large        62  0.161


### Q3. Extract the **title** (Mr, Mrs, Miss, Master, …) out of `name`. Then collapse the
rare ones into `'Rare'`, keeping only titles with at least 10 passengers.
How many survived per title?

<details><summary>hint</summary>

Names look like `Braund, Mr. Owen Harris` — the title sits between the comma and the period. `.str.extract(r',\s*([^\.]+)\.')` with a capture group.

</details>

In [13]:
df['title'] = df['name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
print(df['title'].value_counts(), '\n')

common = df['title'].value_counts()[lambda s: s >= 10].index
df['title'] = df['title'].where(df['title'].isin(common), 'Rare')

print(df.groupby('title')['survived'].agg(n='count', rate='mean').round(3))

# 'Master' (a boy) survives at ~57% vs 'Mr' at ~16%: the title splits the male passengers
# into men and boys, which `sex` alone cannot do. This is the classic Titanic feature.

title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64 

          n   rate
title             
Master   40  0.575
Miss    182  0.698
Mr      517  0.157
Mrs     125  0.792
Rare     27  0.444


### Q4. Earlier you noted `fare` is per-ticket, not per-person. Fix it: build `fare_per_person`
by dividing the fare by the number of passengers sharing that `ticket`.
Does it correlate with survival better than raw `fare`?

<details><summary>hint</summary>

`df['ticket'].value_counts()` gives counts per ticket; `.map()` broadcasts them back onto every row.

</details>

In [14]:
ticket_counts = df['ticket'].map(df['ticket'].value_counts())
df['fare_per_person'] = df['fare'] / ticket_counts

print(df[['fare', 'fare_per_person', 'survived']].corr()['survived'].round(3))
print(df[['fare', 'fare_per_person']].describe().round(2))

# fare_per_person correlates slightly better and is far more interpretable: it's what one
# seat actually cost, i.e. a real wealth proxy rather than wealth x group size.

fare               0.257
fare_per_person    0.255
survived           1.000
Name: survived, dtype: float64
         fare  fare_per_person
count  891.00           891.00
mean    32.20            17.79
std     49.69            21.22
min      0.00             0.00
25%      7.91             7.76
50%     14.45             8.85
75%     31.00            24.29
max    512.33           221.78


---

## Part B — Split FIRST

Everything from here on learns from data. So the split comes before all of it.

### Q5. Define `X` and `y`, then split 80/20 stratified with `random_state=42`.

Careful about what goes into `X`: drop the identifier and the free-text columns you've
already squeezed. Justify each drop.

<details><summary>hint</summary>

`stratify=y` keeps the ~38% survival rate identical in both splits. Always pass it for classification.

</details>

In [15]:
from sklearn.model_selection import train_test_split

drop_cols = [
    'survived',      # the target
    'passengerid',   # an identifier -- pure noise, and a leak if ids were assigned by group
    'name',          # free text; already distilled into `title`
    'ticket',        # high-cardinality id; already used for fare_per_person
    'cabin',         # 77% missing; would become `has_cabin` (see notebook 01)
    'family_size',   # kept as `family_bin` instead
]
X = df.drop(columns=drop_cols)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print('train rate:', y_train.mean().round(3), '| test rate:', y_test.mean().round(3))

(712, 11) (179, 11)
train rate: 0.383 | test rate: 0.385


In [16]:
X_train

,pclass,sex,age,sibsp,parch,fare,embarked,is_alone,family_bin,title,fare_per_person
692,3,male,NaN,0,0,56.4958,S,1,alone,Mr,8.070829
481,2,male,NaN,0,0,0.0000,S,1,alone,Mr,0.000000
527,1,male,NaN,0,0,221.7792,S,1,alone,Mr,221.779200
855,3,female,18.0,0,1,9.3500,S,0,small,Mrs,9.350000
801,2,female,31.0,1,1,26.2500,S,0,small,Mrs,8.750000
...,...,...,...,...,...,...,...,...,...,...,...
359,3,female,NaN,0,0,7.8792,Q,1,alone,Miss,7.879200
258,1,female,35.0,0,0,512.3292,C,1,alone,Miss,170.776400
736,3,female,48.0,1,3,34.3750,S,0,large,Mrs,8.593750
462,1,male,47.0,0,0,38.5000,S,1,alone,Mr,38.500000


### Q6. `age` still has NaNs. Impute it with the (`pclass`, `sex`) group median — but do it
**without leaking**. The group medians must come from the training set only.

<details><summary>hint</summary>

Compute the medians on `X_train`, then apply that same lookup to both frames. The fallback `.fillna(X_train['age'].median())` covers a (pclass, sex) combo that only exists in test.

</details>

In [17]:
medians = X_train.groupby(['pclass', 'sex'])['age'].median()
print(medians)

def fill_age(frame):
    frame = frame.copy()
    keys = list(zip(frame['pclass'], frame['sex']))
    lookup = pd.Series([medians.get(k, np.nan) for k in keys], index=frame.index)
    frame['age'] = frame['age'].fillna(lookup).fillna(X_train['age'].median())
    return frame

X_train = fill_age(X_train)
X_test = fill_age(X_test)   # test rows filled with TRAIN medians -- this is the point

print('NaNs -> train:', X_train['age'].isna().sum(), '| test:', X_test['age'].isna().sum())

pclass  sex   
1       female    35.0
        male      40.0
2       female    29.0
        male      30.0
3       female    21.5
        male      26.0
Name: age, dtype: float64
NaNs -> train: 0 | test: 0


### Q7. Fill the remaining `embarked` NaNs with the training mode.

<details><summary>hint</summary>

Same rule as the median: the mode is a statistic, so it is learned on train.

</details>

In [18]:
mode = X_train['embarked'].mode()[0]
X_train['embarked'] = X_train['embarked'].fillna(mode)
X_test['embarked'] = X_test['embarked'].fillna(mode)
print('mode used:', mode, '| NaNs:', X_train['embarked'].isna().sum(),
      X_test['embarked'].isna().sum())

mode used: S | NaNs: 0 0


---

## Part C — Encoding

### Q8. Encode the nominal columns (`sex`, `embarked`, `title`, `family_bin`) with one-hot.
Use `drop_first=True` and explain what it prevents.
Then align the train and test columns — and explain why that step is necessary.

<details><summary>hint</summary>

`pd.get_dummies(df, columns=[...], drop_first=True)`. Then `X_test.reindex(columns=X_train.columns, fill_value=0)`. In a real pipeline you'd use `OneHotEncoder(handle_unknown='ignore')` and get this for free.

</details>

In [19]:
nominal = ['sex', 'embarked', 'title', 'family_bin']
X_train_enc = pd.get_dummies(X_train, columns=nominal, drop_first=True)
X_test_enc = pd.get_dummies(X_test, columns=nominal, drop_first=True)

# drop_first removes one level per column. Without it the dummies of a column sum to 1
# for every row -- perfectly collinear with the intercept ("dummy variable trap"), which
# makes a linear model's coefficients unidentifiable. Trees don't care either way.

# Aligning matters because get_dummies builds columns from the values *present in that
# frame*. If a rare title appears only in train, test ends up a column short and .predict
# throws. Reindex test onto train's columns:
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print(X_train_enc.shape, X_test_enc.shape)
print(list(X_train_enc.columns))

(712, 16) (179, 16)
['pclass', 'age', 'sibsp', 'parch', 'fare', 'is_alone', 'fare_per_person', 'sex_male', 'embarked_Q', 'embarked_S', 'title_Miss', 'title_Mr', 'title_Mrs', 'title_Rare', 'family_bin_small', 'family_bin_large']


### Q9. Suppose `deck` had 8 levels and `ticket` had 680. Explain, in a comment, why you would
**not** one-hot encode `ticket`, and name two alternatives.

<details><summary>hint</summary>

Think about what 680 columns does to a 891-row dataset.

</details>

In [20]:
print('ticket cardinality:', df['ticket'].nunique(), 'of', len(df), 'rows')

# One-hot on 680 levels would add 680 near-empty columns to a 891-row dataset: more
# features than rows, almost all zeros. Every column would be a memorised id, so the model
# overfits and learns nothing that generalises to an unseen ticket.
# Alternatives:
#   1. Target encoding -- replace each level with the mean target for that level (fit on
#      train only, with smoothing/CV folds, or it leaks the target straight into X).
#   2. Frequency encoding -- replace each level with its count; or extract the structure
#      instead (the ticket prefix, or the group size, as done for fare_per_person).

ticket cardinality: 681 of 891 rows


### Q10. `pclass` is 1/2/3. Should you one-hot it or leave it as an integer? Argue both sides,
then pick one.

<details><summary>hint</summary>

Is it ordered? Yes. Are the gaps between levels equal? Look at the rates.

</details>

In [21]:
print(df.groupby('pclass')['survived'].mean().round(3))

# Leave as int: pclass is genuinely ordinal (1st > 2nd > 3rd) and the survival rates are
# monotonic (.63/.47/.24), so the integer ordering carries real information a linear model
# can use, with one coefficient instead of two.
# One-hot: the *spacing* is fake -- the 1st->2nd gap (.16) and 2nd->3rd gap (.23) are not
# equal, and an int forces a constant per-step effect on a linear model. Trees don't care;
# they threshold and recover the split either way.
# Verdict: keep it as an int here. It's only 3 levels, it's ordered, and it's monotonic in
# the target -- if the rates were non-monotonic, one-hot would be the call.

pclass
1    0.630
2    0.473
3    0.242
Name: survived, dtype: float64


---

## Part D — Scaling & multicollinearity

### Q11. Scale the features with `StandardScaler` for a logistic regression. Fit on train,
transform both. Then show — with a number — that you did not leak.

<details><summary>hint</summary>

`fit_transform` on train, `transform` on test. Then check: is the test mean exactly 0? It shouldn't be.

</details>

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled = scaler.transform(X_test_enc)      # transform only -- never fit on test

print('train mean ~0, std ~1:', X_train_scaled.mean().round(3), X_train_scaled.std().round(3))
print('test  mean       :', X_test_scaled.mean().round(3))

# The test mean is close to 0 but NOT exactly 0 -- and that is the proof. If it were
# exactly 0 the scaler would have seen the test set's own mean, i.e. leakage.

train mean ~0, std ~1: 0.0 1.0
test  mean       : 0.013


### Q12. Would you scale for a `RandomForestClassifier`? Explain in one sentence, and say
whether scaling would *hurt* if you did it anyway.

<details><summary>hint</summary>

What does a tree actually do with a feature's value?

</details>

In [23]:
# No. A tree splits on thresholds ("age < 12"), and any monotonic rescaling maps that
# threshold to an equivalent one -- the tree structure is unchanged. It wouldn't hurt
# accuracy, it's just wasted compute and one more fitted object to keep in sync at
# serving time. Scaling matters for models that use distances or coefficient magnitudes:
# KNN, SVM, linear/logistic regression, PCA, neural nets.

### Q13. Compute the VIF for the numeric features. Is anything above the usual threshold of 10?
Whatever the answer, find the redundant pair — VIF is not the only tool you have.

<details><summary>hint</summary>

`variance_inflation_factor(X.values, i)` for each column index. Cast to float — it chokes on bool dummies. Then check `.corr()` on the pair you suspect and see whether the two tools agree.

</details>

In [24]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare', 'fare_per_person', 'is_alone']
Xv = X_train_enc[num_cols].astype(float)

vif = pd.DataFrame({
    'feature': num_cols,
    'VIF': [variance_inflation_factor(Xv.values, i) for i in range(Xv.shape[1])],
}).sort_values('VIF', ascending=False)
display(vif.round(2))

print('\ncorr(fare, fare_per_person) =',
      round(X_train_enc[['fare', 'fare_per_person']].corr().iloc[0, 1], 3))

# NOTHING exceeds 10 -- the highest are fare_per_person (~6.6), pclass (~5.9) and
# fare (~5.6). So a rule-following answer would be "no multicollinearity, move on".
# That would be wrong, and this is the point of the question:
#   - fare and fare_per_person correlate at 0.84 and one is derived from the other. The
#     plain correlation catches what the VIF threshold waves through.
#   - VIF here is computed WITHOUT an intercept (statsmodels doesn't add one), which
#     distorts the absolute values -- with a constant column added they'd read differently.
#     So treat the ranking as the signal, not the number against a magic threshold.
# Thresholds are a starting point, not the finding.

,feature,VIF
5,fare_per_person,6.30
0,pclass,6.09
4,fare,5.56
6,is_alone,5.25
1,age,4.93
2,sibsp,1.94
3,parch,1.91



corr(fare, fare_per_person) = 0.841


### Q14. Resolve the collinearity you found. Then state which models actually needed you to
bother.

<details><summary>hint</summary>

Drop one of each redundant pair, keeping the more interpretable one.

</details>

In [25]:
X_train_final = X_train_enc.drop(columns=['fare', 'sibsp', 'parch'])
X_test_final = X_test_enc.drop(columns=['fare', 'sibsp', 'parch'])

# Kept fare_per_person over fare (better-behaved, more interpretable) and family_bin +
# is_alone over the raw sibsp/parch counts. One representative per idea.
print(list(X_train_final.columns))

# Who needed this: plain logistic/linear regression, where collinearity makes coefficients
# unstable and their signs unreadable -- fatal if the coefficient is the deliverable.
# Who didn't: Random Forest, XGBoost (they just split on whichever correlate they see
# first -- it dilutes feature importances but not accuracy), and Ridge/Lasso, which are
# built to absorb exactly this.

['pclass', 'age', 'is_alone', 'fare_per_person', 'sex_male', 'embarked_Q', 'embarked_S', 'title_Miss', 'title_Mr', 'title_Mrs', 'title_Rare', 'family_bin_small', 'family_bin_large']


---

## Part E — Do it properly

### Q15. Everything above was done by hand and is a pain to reproduce at serving time. Rebuild
it as a single `ColumnTransformer` + `Pipeline` that takes raw columns in and gives
predictions out. Cross-validate it.

This is the answer they want when they ask *"how would you productionise this?"*

<details><summary>hint</summary>

`ColumnTransformer([('num', num_pipe, numeric_cols), ('cat', cat_pipe, cat_cols)])`, then wrap it with the classifier in a `Pipeline`. Use `OneHotEncoder(handle_unknown='ignore')` so an unseen category at predict time doesn't crash you.

</details>

In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

numeric = ['pclass', 'age', 'fare_per_person', 'is_alone']
categorical = ['sex', 'embarked', 'title', 'family_bin']

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', drop='first')),
])

pre = ColumnTransformer([
    ('num', numeric_pipe, numeric),
    ('cat', categorical_pipe, categorical),
])
pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X[numeric + categorical], y, cv=cv, scoring='accuracy')
print(f'CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}')

pipe.fit(X_train[numeric + categorical], y_train)
print('test accuracy:', pipe.score(X_test[numeric + categorical], y_test))

# Why this is the real answer: every fold refits the imputer, scaler and encoder on that
# fold's training rows only. Leakage becomes structurally impossible rather than something
# you have to remember. And the whole object pickles as one artefact -- no risk of the
# scaler drifting out of sync with the model in production.

CV accuracy: 0.828 +/- 0.003
test accuracy: 0.8212290502793296


---

## Debrief

The five leakage traps in this notebook, in the order people fall into them:

1. **Imputing before splitting** — the median you fill test with was computed using test.
2. **`fit_transform` on the test set** — the scaler learns the test distribution.
3. **Target encoding without CV folds** — puts the answer directly into a feature.
4. **Feature selection on the full dataset** — you chose the features using test.
5. **Tuning against the test set** — after the first peek, it isn't a test set any more.

If you build a `Pipeline`, 1–4 stop being possible. Only #5 stays your problem.